# LGR5 Marker-Domain Analysis

Train one or more GIN models and analyze predicted curvature profiles around connected LGR5-positive domains.

In [ ]:
import copy
import inspect
import json
import math
import pickle
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "training_data"

sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("torch        =", torch.__version__)
print("cuda         =", torch.cuda.is_available())

In [ ]:
# Data and target settings
DATASET_NAME = "mean_curvature_smooth"
TARGET_INDICES = [0]
TARGET_INDEX_FOR_ANALYSIS = 0
USE_GLOBAL_FEATURES = True

# Filtering and preprocessing settings
MISSING_COMPLEXITY_GROUP = {
    "dataset": "20251201",
    "timepoint": "day4p5",
    "fill_value": 2.1,
}
SPHERICITY_MAX = 0.92
SPHERICAL_MARKER_DIVERSITY_MIN = 0.5
COMPLEXITY_MIN = 2.0
INTERPOLATE_TARGET_OUTLIERS = True
OUTLIER_CLIP_QUANTILES = (0.005, 0.995)

# Split settings
VAL_FRAC = 0.2
SPLIT_SEED = None
FORCED_VAL_KEYS = set()

# Model settings
DEPTHS = [2, 4]
HIDDEN_DIM = 4 * 64
DROPOUT = 0.1
NORM = "batch"
RESIDUAL = True

# Training settings
LR = 3e-4
BATCH_SIZE = 128
MAX_EPOCHS = 2000
PATIENCE = 30
NUM_WORKERS = 4
EDGE_LOSS_WEIGHT = 0.20
EDGE_LOSS_PARAMS = {
    "weighted": False,
    "alpha": 2.0,
    "normalize_by": "graph_std",
    "clip_weight": 4.0,
}

# Marker-domain analysis settings
DOMAIN_MARKER = "LGR5"
MARKER_THRESHOLD = 0.5
MAX_OUTSIDE_DISTANCE = 4
MAX_INSIDE_DISTANCE = None
SIZE_BINS = list(range(1, 9))
OVERFLOW_SIZE_BIN = True
OVERFLOW_LABEL = "8+"
RUN_SHRINK = True
SHRINK_TARGET_SIZES = None
INCLUDE_ALL_INTERMEDIATE_SHRINK_SIZES = False
MAX_COMPONENTS_FOR_SHRINK = 200
DOMAIN_BATCH_SIZE = 64

## Load And Filter Graphs

In [ ]:
from src.data.io import load_graph_dataset_from_dir, select_graph_targets
from src.data.metadata import (
    attach_metadata_to_graphs,
    load_aux_metadata_for_dir,
    load_marker_names_from_dir,
    print_graph_and_metadata_fields,
)

data_dir = DATA_ROOT / DATASET_NAME
graphs = load_graph_dataset_from_dir(str(data_dir))
print(f"Loaded {len(graphs)} organoids.")
if graphs:
    print("Raw y shape:", tuple(graphs[0].y.shape))

meta = load_aux_metadata_for_dir(str(data_dir))
attached = attach_metadata_to_graphs(graphs, meta, exclude_keys=None)
print(f"Attached metadata to {attached}/{len(graphs)} graphs.")

graphs = select_graph_targets(graphs, target_indices=TARGET_INDICES, inplace=False)
if graphs:
    print("Selected y shape:", tuple(graphs[0].y.shape))

marker_names = load_marker_names_from_dir(str(data_dir))
if marker_names is None:
    n_markers = int(graphs[0].x.size(1)) if graphs else 0
    marker_names = [f"marker_{i}" for i in range(n_markers)]
print(f"Loaded {len(marker_names)} markers:", marker_names)

print_graph_and_metadata_fields(graphs)

In [ ]:
from src.data.metadata import fill_missing_metadata_for_group
from src.data.filters import (
    filter_graphs_by_marker_diversity,
    filter_graphs_by_numeric_metadata,
    filter_graphs_by_sphericity,
)
from src.data.preprocessing import interpolate_target_outliers_from_neighbors

graphs = fill_missing_metadata_for_group(
    graphs,
    field="complexity",
    fill_value=MISSING_COMPLEXITY_GROUP["fill_value"],
    dataset=MISSING_COMPLEXITY_GROUP["dataset"],
    timepoint=MISSING_COMPLEXITY_GROUP["timepoint"],
)

graphs, g_spherical = filter_graphs_by_sphericity(
    graphs,
    max_sphericity=SPHERICITY_MAX,
    print_summary=True,
    return_rejected=True,
)

g_spherical = filter_graphs_by_marker_diversity(
    g_spherical,
    min_score=SPHERICAL_MARKER_DIVERSITY_MIN,
    print_summary=True,
)

graphs = filter_graphs_by_numeric_metadata(
    graphs,
    key="complexity",
    min_value=COMPLEXITY_MIN,
    allow_missing=False,
    inplace=False,
    print_summary=True,
)

graphs = graphs + g_spherical
print(f"After filtering and spherical rescue: {len(graphs)} organoids.")

if INTERPOLATE_TARGET_OUTLIERS:
    graphs, outlier_info = interpolate_target_outliers_from_neighbors(
        graphs,
        target_indices=None,
        clip_quantiles=OUTLIER_CLIP_QUANTILES,
    )
else:
    outlier_info = None

In [ ]:
from src.data.metadata import add_log_metadata_features, promote_metadata_to_graph_tensors

field_specs = [
    {
        "meta_keys": [
            "log_surface_area",
            "log_volume",
            "log_volume_over_area",
            "log_num_cells",
        ],
        "attr_name": "global_feat",
        "kind": "graph_vector",
        "dtype": torch.float32,
    },
]

if USE_GLOBAL_FEATURES:
    graphs = add_log_metadata_features(graphs, inplace=False)
    graphs = promote_metadata_to_graph_tensors(graphs, field_specs, inplace=False)
    print("Promoted metadata fields to graph tensor attributes.")
else:
    print("Global features disabled; no global_feat attribute was attached.")

In [ ]:
from src.data.splits import graph_metadata_key, train_val_split_graphs
from src.data.metadata import infer_global_dim, snapshot_graph_metadata, strip_graph_metadata
from src.data.target_transforms import AsinhStandardizeTransform, standardize_graph_global_features

g_train, g_val, split_info = train_val_split_graphs(
    graphs,
    val_frac=VAL_FRAC,
    seed=SPLIT_SEED,
    force_val_keys=FORCED_VAL_KEYS,
    key_fn=graph_metadata_key,
)
print(f"Split -> train: {len(g_train)} | val: {len(g_val)}")

val_meta_lookup = snapshot_graph_metadata(g_val)

g_train = strip_graph_metadata(g_train, inplace=False)
g_val = strip_graph_metadata(g_val, inplace=False)

target_transform = AsinhStandardizeTransform(robust=True).fit(g_train)
target_transform.transform_graphs(g_train)
target_transform.transform_graphs(g_val)

center_global, scale_global = None, None
if USE_GLOBAL_FEATURES:
    center_global, scale_global = standardize_graph_global_features(
        g_train,
        g_val,
        attr_name="global_feat",
        robust=False,
    )

global_dim = infer_global_dim(g_train)
print("global_dim =", global_dim)
print("target_transform =", target_transform.name)

## Train GIN Models

In [ ]:
from src.models.gnn import GINCurvature
from src.training.loop import TrainConfig, train
from src.training.losses import WeightedLossTerm, edge_loss_term

aux_losses = [
    WeightedLossTerm(
        name="edge",
        fn=edge_loss_term,
        weight=EDGE_LOSS_WEIGHT,
        params=EDGE_LOSS_PARAMS,
    ),
]

cfg = TrainConfig(
    lr=LR,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    aux_losses=aux_losses,
)

device = cfg.device if "cfg" in globals() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

In [ ]:
trained_models = {}
training_logs = {}

for depth in DEPTHS:
    print("\n" + "=" * 80)
    print(f"Training GINCurvature depth={depth}")

    model = GINCurvature(
        n_markers=int(g_train[0].x.size(1)),
        global_dim=infer_global_dim(g_train),
        hidden_dim=HIDDEN_DIM,
        num_layers=int(depth),
        dropout=DROPOUT,
        residual=RESIDUAL,
        norm=NORM,
    )

    model, metrics, history = train(model, g_train, g_val, cfg)
    trained_models[int(depth)] = model
    training_logs[int(depth)] = {
        "metrics": metrics,
        "history": history,
    }

print("Finished training depths:", list(trained_models))

## LGR5 Domain Analysis

In [ ]:
from src.analysis.marker_domain import (
    marker_index_from_name,
    run_marker_domain_profile_analysis,
    size_bin_order_from_spec,
    plot_domain_profiles,
    plot_observed_vs_shrink_profiles,
    plot_shrink_delta_profiles,
)

# Resolve robustly in case the loaded marker names differ in capitalization.
try:
    domain_marker_index = marker_index_from_name(DOMAIN_MARKER, marker_names)
    domain_marker_name = marker_names[domain_marker_index]
except ValueError:
    lower_lookup = {name.lower(): i for i, name in enumerate(marker_names)}
    key = str(DOMAIN_MARKER).lower()
    if key not in lower_lookup:
        raise
    domain_marker_index = lower_lookup[key]
    domain_marker_name = marker_names[domain_marker_index]

print(f"Analyzing marker {domain_marker_name!r} at feature column {domain_marker_index}.")

n_positive_by_graph = [int((g.x[:, domain_marker_index] > MARKER_THRESHOLD).sum().item()) for g in g_val]
print("Validation graphs with positive cells:", sum(n > 0 for n in n_positive_by_graph), "/", len(g_val))
print("Total positive validation cells:", sum(n_positive_by_graph))

In [ ]:
domain_results_by_depth = {}

for depth, model in trained_models.items():
    print("\n" + "=" * 80)
    print(f"Marker-domain analysis for {domain_marker_name} | depth={depth}")

    result = run_marker_domain_profile_analysis(
        graphs=g_val,
        model=model,
        marker=domain_marker_index,
        marker_names=marker_names,
        marker_threshold=MARKER_THRESHOLD,
        max_outside_distance=MAX_OUTSIDE_DISTANCE,
        max_inside_distance=MAX_INSIDE_DISTANCE,
        size_bins=SIZE_BINS,
        overflow_size_bin=OVERFLOW_SIZE_BIN,
        overflow_label=OVERFLOW_LABEL,
        run_shrink=RUN_SHRINK,
        shrink_target_sizes=SHRINK_TARGET_SIZES,
        include_all_intermediate_shrink_sizes=INCLUDE_ALL_INTERMEDIATE_SHRINK_SIZES,
        max_components_for_shrink=MAX_COMPONENTS_FOR_SHRINK,
        device=device,
        batch_size=DOMAIN_BATCH_SIZE,
        target_index=TARGET_INDEX_FOR_ANALYSIS,
    )
    domain_results_by_depth[int(depth)] = result

    observed_df = result["observed_df"]
    shrink_df = result.get("shrink_df", pd.DataFrame())
    print("observed rows:", len(observed_df), "components:", len(result["components"]))
    print("shrink rows:", len(shrink_df))

In [ ]:
component_rows = []
for depth, result in domain_results_by_depth.items():
    for comp in result["components"]:
        row = {"depth": depth, "graph_index": comp.graph_index, "component_id": comp.component_id}
        row.update(comp.stats)
        component_rows.append(row)

component_df = pd.DataFrame(component_rows)
if component_df.empty:
    print("No LGR5-positive components found.")
else:
    display(component_df.groupby("depth")["cluster_size"].describe())
    display(component_df.head())

## Plot Results

In [ ]:
if not component_df.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    bins = np.arange(1, component_df["cluster_size"].max() + 2) - 0.5
    for depth, sub in component_df.groupby("depth"):
        ax.hist(sub["cluster_size"], bins=bins, alpha=0.45, label=f"depth {depth}")
    ax.set_xlabel(f"{domain_marker_name}+ component size")
    ax.set_ylabel("count")
    ax.set_title(f"{domain_marker_name}+ domain size distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
n_depths = len(domain_results_by_depth)
fig, axes = plt.subplots(1, n_depths, figsize=(7.5 * n_depths, 4.8), squeeze=False, sharey=True)
axes = axes.ravel()

for ax, (depth, result) in zip(axes, domain_results_by_depth.items()):
    observed_df = result["observed_df"]
    if observed_df.empty:
        ax.set_title(f"depth={depth}: no domains")
        ax.axis("off")
        continue
    plot_domain_profiles(
        observed_df,
        value_col="mu",
        condition="observed",
        size_bins=SIZE_BINS,
        overflow_size_bin=OVERFLOW_SIZE_BIN,
        overflow_label=OVERFLOW_LABEL,
        ax=ax,
        title=f"Observed {domain_marker_name}+ domains | depth={depth}",
        ylabel="predicted curvature",
    )

plt.tight_layout()
plt.show()

In [ ]:
if RUN_SHRINK:
    n_depths = len(domain_results_by_depth)
    fig, axes = plt.subplots(n_depths, 2, figsize=(14, 4.8 * n_depths), squeeze=False, sharey="row")

    for row, (depth, result) in enumerate(domain_results_by_depth.items()):
        observed_df = result["observed_df"]
        shrink_df = result.get("shrink_df", pd.DataFrame())
        if observed_df.empty or shrink_df.empty:
            axes[row, 0].set_title(f"depth={depth}: insufficient shrink data")
            axes[row, 0].axis("off")
            axes[row, 1].axis("off")
            continue
        plot_observed_vs_shrink_profiles(
            observed_df,
            shrink_df,
            value_col="mu",
            size_bins=SIZE_BINS,
            overflow_size_bin=OVERFLOW_SIZE_BIN,
            overflow_label=OVERFLOW_LABEL,
            axes=axes[row],
        )
        axes[row, 0].set_title(f"Observed | depth={depth}")
        axes[row, 1].set_title(f"Shrink-only | depth={depth}")

    plt.tight_layout()
    plt.show()

In [ ]:
if RUN_SHRINK:
    n_depths = len(domain_results_by_depth)
    fig, axes = plt.subplots(1, n_depths, figsize=(7.5 * n_depths, 4.8), squeeze=False, sharey=True)
    axes = axes.ravel()

    for ax, (depth, result) in zip(axes, domain_results_by_depth.items()):
        shrink_df = result.get("shrink_df", pd.DataFrame())
        if shrink_df.empty:
            ax.set_title(f"depth={depth}: no shrink rows")
            ax.axis("off")
            continue
        plot_shrink_delta_profiles(
            shrink_df,
            value_col="delta_mu",
            size_bins=SIZE_BINS,
            overflow_size_bin=OVERFLOW_SIZE_BIN,
            overflow_label=OVERFLOW_LABEL,
            ax=ax,
            title=f"Shrink-only Δ prediction | depth={depth}",
        )

    plt.tight_layout()
    plt.show()

In [ ]:
summary_rows = []
for depth, result in domain_results_by_depth.items():
    obs = result["observed_summary"].copy()
    if not obs.empty:
        obs["depth"] = depth
        obs["quantity"] = "mu"
        summary_rows.append(obs)

    shrink_delta = result.get("shrink_summary_delta_mu", pd.DataFrame()).copy()
    if not shrink_delta.empty:
        shrink_delta["depth"] = depth
        shrink_delta["quantity"] = "delta_mu"
        summary_rows.append(shrink_delta)

summary_df = pd.concat(summary_rows, ignore_index=True) if summary_rows else pd.DataFrame()
display(summary_df.head(20))